### Imports

In [0]:
# ============================================================================
# SMART MANUFACTURING INTELLIGENCE PLATFORM (SMIP)
# Gold Layer
#
# Notebook : 03_oee_summary
# Layer    : Gold
#
# Description
# ----------------------------------------------------------------------------
# Calculates Operational Excellence KPIs (Availability, Performance,
# Quality and OEE) from Silver Fact tables.
#
# Grain
# ----------------------------------------------------------------------------
# Production Date
# +
# Machine
# +
# Planned Shift
# ============================================================================

from pyspark.sql.functions import (
    col,
    countDistinct,
    count,
    avg,
    min,
    max,
    sum,
    round,
    when,
    year,
    quarter,
    month,
    weekofyear,
    to_date,
    date_format,
    unix_timestamp,
    expr,
    lag,
    dense_rank
)

from pyspark.sql.window import Window

from framework.core.session import spark

from framework.core.configuration import (
    SILVER_LAYER,
    GOLD_LAYER
)

from framework.core.logger import (
    banner,
    info,
    success
)

from framework.io.delta import write_delta

### Start Notebook

In [0]:
banner("Gold Layer - OEE Summary")

### Read Silver Tables

In [0]:
press = (

    spark.table(f"{SILVER_LAYER}.fact_press_operations")

    .select(

        "press_operation_key",

        "execution_id",

        "machine_key",

        "factory_key",

        "serial_number",

        "operation_start",

        "operation_end",

        "cycle_time_sec",

        "target_force_kn",

        "actual_force_kn",

        "force_deviation_kn"

    )

)

production = (

    spark.table(f"{SILVER_LAYER}.fact_production")

    .select(

        "execution_id",

        "planned_shift"

    )

)

quality = (

    spark.table(f"{SILVER_LAYER}.fact_quality")

    .select(

        "execution_id",

        "serial_number",

        "result"

    )

)

machines = (

    spark.table(f"{SILVER_LAYER}.dim_machines")

    .select(

        "machine_key",

        "machine_id",

        "machine_name",

        "machine_type"

    )

)

factory = (

    spark.table(f"{SILVER_LAYER}.dim_factory")

    .select(

        "factory_key",

        "hall_name",

        "line_name"

    )

)

info("Silver tables loaded.")

### Quality Summary

In [0]:
quality_summary = (

    quality

    .groupBy(

        "execution_id"

    )

    .agg(

        count("*").alias("tests_performed"),

        sum(

            when(

                col("result")=="PASS",

                1

            ).otherwise(0)

        ).alias("good_parts")

    )

)

### Join Everything

In [0]:
df = (

    press

    .join(

        production,

        "execution_id",

        "left"

    )

    .join(

        quality_summary,

        "execution_id",

        "left"

    )

    .join(

        machines,

        "machine_key",

        "left"

    )

    .join(

        factory,

        "factory_key",

        "left"

    )

)

### Calendar Attributes

In [0]:
df = (

    df

    .withColumn(

        "production_date",

        to_date("operation_start")

    )

    .withColumn(

        "production_year",

        year("operation_start")

    )

    .withColumn(

        "production_quarter",

        quarter("operation_start")

    )

    .withColumn(

        "production_month",

        month("operation_start")

    )

    .withColumn(

        "production_week",

        weekofyear("operation_start")

    )

    .withColumn(

        "production_day",

        date_format(

            "operation_start",

            "EEEE"

        )

    )

)

### Aggregate

In [0]:
oee = (

    df

    .groupBy(

        "production_date",

        "production_year",

        "production_quarter",

        "production_month",

        "production_week",

        "production_day",

        "planned_shift",

        "machine_key",

        "machine_id",

        "machine_name",

        "machine_type",

        "hall_name",

        "line_name"

    )

    .agg(

        countDistinct("serial_number").alias("units_produced"),

        count("serial_number").alias("operations"),

        avg("cycle_time_sec").alias("average_cycle_time_sec"),

        min("cycle_time_sec").alias("ideal_cycle_time_sec"),

        sum("cycle_time_sec").alias("operating_time_sec"),

        min(

            unix_timestamp("operation_start")

        ).alias("shift_start"),

        max(

            unix_timestamp("operation_end")

        ).alias("shift_end"),

        avg("actual_force_kn").alias("average_force_kn"),

        avg("force_deviation_kn").alias("average_force_deviation_kn"),

        max("tests_performed").alias("tests_performed"),

        max("good_parts").alias("good_parts")

    )

)

### KPI Calculations

In [0]:
oee = (

    oee

    .withColumn(

        "available_time_sec",

        col("shift_end")-col("shift_start")

    )

    .withColumn(

        "availability_pct",

        round(

            col("operating_time_sec")

            *100

            /col("available_time_sec"),

            2

        )

    )

    .withColumn(

        "performance_pct",

        round(

            col("ideal_cycle_time_sec")

            *100

            /col("average_cycle_time_sec"),

            2

        )

    )

    .withColumn(

        "quality_pct",

        round(

            col("good_parts")

            *100

            /col("tests_performed"),

            2

        )

    )

    .withColumn(

        "daily_oee",

        round(

            (

                col("availability_pct")

                *

                col("performance_pct")

                *

                col("quality_pct")

            )/10000,

            2

        )

    )

)

### Window Analytics

In [0]:
machine_window = (

    Window

    .partitionBy(

        "machine_id"

    )

    .orderBy(

        "production_date"

    )

)

ranking_window = (

    Window

    .partitionBy(

        "production_date"

    )

    .orderBy(

        col("daily_oee").desc()

    )

)

### Trends

In [0]:
oee = (

    oee

    .withColumn(

        "rolling_7_day_oee",

        round(

            avg(

                "daily_oee"

            ).over(

                machine_window.rowsBetween(-6,0)

            ),

            2

        )

    )

    .withColumn(

        "rolling_30_day_oee",

        round(

            avg(

                "daily_oee"

            ).over(

                machine_window.rowsBetween(-29,0)

            ),

            2

        )

    )

    .withColumn(

        "previous_day_oee",

        lag(

            "daily_oee"

        ).over(

            machine_window

        )

    )

    .withColumn(

        "oee_change",

        round(

            col("daily_oee")

            -

            col("previous_day_oee"),

            2

        )

    )

    .withColumn(

        "machine_rank",

        dense_rank().over(

            ranking_window

        )

    )

)

### Write Gold

In [0]:
write_delta(

    oee,

    f"{GOLD_LAYER}.oee_summary"

)

success("gold.oee_summary created successfully.")

### Validation

In [0]:
display(oee)

display(

    spark.sql(f"""

    SELECT

        COUNT(*) rows

    FROM {GOLD_LAYER}.oee_summary

    """)

)

display(

    spark.sql(f"""

    SELECT

        production_date,

        ROUND(

            AVG(daily_oee),

            2

        ) AS plant_oee

    FROM {GOLD_LAYER}.oee_summary

    GROUP BY production_date

    ORDER BY production_date

    """)

)

display(

    spark.sql(f"""

    SELECT

        machine_name,

        ROUND(

            AVG(daily_oee),

            2

        ) AS average_oee

    FROM {GOLD_LAYER}.oee_summary

    GROUP BY machine_name

    ORDER BY average_oee DESC

    """)

)


display(

    spark.sql(f"""

    SELECT

        planned_shift,

        ROUND(

            AVG(daily_oee),

            2

        ) AS shift_oee

    FROM {GOLD_LAYER}.oee_summary

    GROUP BY planned_shift

    ORDER BY shift_oee DESC

    """)

)